In [1]:
import rdworks
import math
import numpy as np
import pandas as pd
import altair as alt

from rdworks import StateNetwork

rdworks.__version__

'0.70.0'

In [2]:
def line_plot(data: pd.DataFrame,
              x_label: str | None = None,
              y_label: str | None = None,
              c_label: str | None = None,
              x_data: str | None = None,
              y_data: str | None = None,
              c_data: str | None = None,
              width: int = 600, 
              height: int = 300, 
              palette: str = 'tableau10'):
    
    df = pd.DataFrame(data)
    
    if x_data is None:
        x_data = 'X'
        assert x_data in df.columns
        
    if y_data is None:
        y_data = 'Y'
        assert y_data in df.columns
        
    if c_data is None:
        if 'C' in df.columns:
            c_data = 'C'
        else:
            c_data = None
            
    if x_label is None:
        x_label = x_data
    
    if y_label is None:
        y_label = y_data
    
    if c_label is None:
        c_label = c_data
    
    if c_data is not None:
        # line plot
        lineplot = alt.Chart(df).mark_line().encode(
            x=alt.X(f'{x_data}:Q', title=x_label),
            y=alt.Y(f'{y_data}:Q', title=y_label),
            color=alt.Color(f'{c_data}:N', 
                            scale=alt.Scale(scheme=palette),
                            legend=alt.Legend(
                                title=c_label,
                                orient='right',
                                columns=1,
                                symbolFillColor='gray',
                                labelColor='darkblue'
                            ),
                           ),
            
        ).properties(width=width, height=height)
        
        # data labels
        labels = alt.Chart(df).mark_text(align='left', dx=5, dy=-5
        ).encode(
            x=alt.X(f'{x_data}:Q', aggregate={'argmax': f'{y_data}'}),
            y=alt.Y(f'{y_data}:Q', aggregate={'argmax': f'{y_data}'}),
            text=f'{c_data}:N',
            color=f'{c_data}:N',
        )

        return lineplot + labels
    
    else:
        lineplot = alt.Chart(df).mark_line().encode(
            x=alt.X(f'{x_data}:Q', title=x_label),
            y=alt.Y(f'{y_data}:Q', title=y_label),
        ).properties(width=width, height=height)
        
        return lineplot

# Build State Network

In [3]:
smiles = 'c1ccc(CNc2ncnc3ccccc23)cc1'
sn = StateNetwork()
sn.build(smiles=smiles, max_formal_charge=3, tautomer_rule=None)
assert len(sn.visited_states) == 12
assert len(sn.graph.nodes()) == 12
se = sn.get_state_ensemble()

# set energies

In [4]:
# dG calculated from Uni-pKa model (energies at reference pH, 6.504894871171601) 
dG = np.array([
    -6.025253772735596, -2.9201512336730957, -2.7405877113342285,
    -2.9639060497283936, 7.656927108764648,  19.67357063293457,
    21.269811630249023, 11.911577224731445,   7.5623698234558105,
    10.144123077392578, 21.36874008178711,   12.132856369018555])

se.set_energies(dG, ref_ph=6.504894871171601)

ph_values = np.linspace(-2, 16, 180)
p = se.get_population(ph_values, C=math.log(10), beta=1.0)

## pH vs microstate population

In [5]:

plotdata1, populated_state_idx = se.get_plot_data_pH_vs_microstate_population(ph_values, p, threshold=0.0)
line_plot(plotdata1, x_data='pH', y_data='p', c_data='microstate')

alt.LayerChart(...)

## pH vs site_charge (micro-pKa)

In [6]:
plotdata2, inflection_points_micro = se.get_plot_data_pH_vs_site_charge(ph_values, p)
line_plot(plotdata2, x_data='pH', y_data='charge', c_data='site')

alt.LayerChart(...)

In [7]:
inflection_points_micro

{5: [(0.9889698668041688, 0.6688314952448096),
  (5.615688277356292, 0.17360523570635875),
  (12.446885025801043, -0.49995825197554133)],
 7: [(1.0063092640075326, 0.6477305086053503),
  (5.615669988657333, 0.14507544526089083)],
 9: [(1.0063082535186254, 0.6837459212331605),
  (5.6156732515409535, 0.18137445929001603)]}

## pH vs mean_charge (macro-pKa)

In [8]:
plotdata3, inflection_points_macro = se.get_plot_data_pH_vs_mean_charge(ph_values, p)
line_plot(plotdata3, x_data='pH', y_data='charge')

alt.Chart(...)

In [9]:
inflection_points_macro

[(1.0006170763037052, 2.000342354869194),
 (5.615677521723759, 0.500055140165614),
 (12.446884560294487, -0.49995726282907393)]